## Installs and Imports

In [ ]:
!pip install -U torch torchvision
!pip install transformers datasets tqdm pandas scipy

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
!pip install --force-reinstall --no-cache-dir typing_extensions==4.11.0
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
## Sometimes needed in runpod to make sure it goes to the network volumne
import os

os.environ["HF_DATASETS_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"
os.environ["HF_HOME"] = "/workspace/hf_home"

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import ViTForImageClassification
from datasets import load_from_disk, concatenate_datasets
import numpy as np
import copy
from collections import defaultdict
import pandas as pd

## Configuration

In [ ]:
num_class = [47, 10, 43, 10, 45, 196, 397, 10]
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]
model_name = "DeiT"
device = "cuda" if torch.cuda.is_available() else "cpu"
refining_type = "Standard" # "Standard" | "Increment_Training"
refiner = "Reverse_Probe" # "Fine_Tuned" | "Linear_Probe" | "Reverse_Probe"

#  https://huggingface.co/facebook/deit-tiny-patch16-224
refer = ViTForImageClassification.from_pretrained("facebook/deit-tiny-patch16-224")
embedding_space_dimension = refer.config.hidden_size

## Data Prep

In [ ]:
# Only if needed
from transformers import DeiTImageProcessor
from datasets import load_dataset

processor = DeiTImageProcessor.from_pretrained('facebook/deit-tiny-patch16-224')

dataset = load_dataset('', cache_dir="/workspace/.hf_cache") # Change This

split = dataset["train"].train_test_split(test_size=0.2, seed=66)

train = split["train"]
val = split["test"]
test = dataset["test"]

def transform_example(batch):
    images = batch["image"]
    images = [img.convert("RGB") if img.mode != "RGB" else img for img in images]
    batch["pixel_values"] = processor(images, return_tensors="np")["pixel_values"]
    return batch

train = train.map(transform_example, batched=True, batch_size=64, num_proc=10)
val = val.map(transform_example, batched=True, batch_size=64, num_proc=10)
test = test.map(transform_example, batched=True, batch_size=64, num_proc=10)

train.save_to_disk(f'/workspace/preprocessed/{dataset_name}/train_processed')
val.save_to_disk(f'/workspace/preprocessed/{dataset_name}/val_processed')
test.save_to_disk(f'/workspace/preprocessed/{dataset_name}/test_processed')

## Class Prep

In [ ]:
# https://github.com/huggingface/transformers/blob/main/src/transformers/models/deit/image_processing_deit.py
class Augmented(torch.nn.Module):
    def __init__(self, model, num_classes, embedding_space_dimension, classifier=None, transform_layer=-1, W=None):
        super().__init__()
        self.model = model
        self.num_classes = num_classes
        self.embedding_space_dimension = embedding_space_dimension
        self.classifier = classifier if classifier is not None else torch.nn.Linear(self.embedding_space_dimension, self.num_classes)
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
        self.transform_layer = transform_layer
    
    def forward(self, images):
        hidden_states = self.model.vit.embeddings(images)

        for i, layer_module in enumerate(self.model.vit.encoder.layer):
            layer_outputs = layer_module(hidden_states)
            hidden_states = layer_outputs # hidden_states = layer_outputs[0]
            if i == self.transform_layer:
                if self.W is None:
                    self.W = torch.eye(hidden_states.shape[-1], device=hidden_states.device, dtype=hidden_states.dtype)
                cls = hidden_states[:, 0, :]
                cls = cls @ self.W
                hidden_states[:, 0, :] = cls
                break
            
            cls = hidden_states[:, 0, :]
        
        hidden_states = self.model.vit.layernorm(hidden_states)
        logits = self.classifier(hidden_states[:, 0, :])

        return logits, cls

## Fine-Tune Prep

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
EPOCHS = 500

In [ ]:
for name in range(0, len(dataset_name)): # len(dataset_name)
  model_path = f"./Models/{model_name}/{refining_type}/{refiner}/{dataset_name[name]}"
  os.makedirs(model_path, exist_ok=True)

  num_classes = num_class[name]
  # Dataset Prep
  train = load_from_disk(f'/workspace/preprocessed/{dataset_name[name]}/train_processed')
  val = load_from_disk(f'/workspace/preprocessed/{dataset_name[name]}/val_processed')
  test = load_from_disk(f'/workspace/preprocessed/{dataset_name[name]}/test_processed')

  train.set_format(type='torch', columns=["image", "label", "pixel_values"])
  val.set_format(type='torch', columns=["image", "label", "pixel_values"])
  test.set_format(type='torch', columns=["image", "label", "pixel_values"])

  def collate_fn(batch):
      images = torch.stack([example["pixel_values"] for example in batch])
      labels = torch.tensor([example["label"] for example in batch])
      
      return {
          "pixel_values": images,
          "labels": labels
      }

  train_loader = DataLoader(train, batch_size=8, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
  val_loader = DataLoader(val, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
  test_loader = DataLoader(test, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

  if refining_type in ["Increment_Training"]:
      labels = train["label"]

      label_to_indices = defaultdict(list)

      for idx, label in enumerate(labels):
          label = int(label)
          label_to_indices[label].append(idx)

      filtered_train = {
          label: train.select(indices) for label, indices in label_to_indices.items()
      }

      train_subsets = {}
      for i in range(num_classes):
          increments = len(filtered_train[i]) // 5
          train_subsets[i] = []
          starter = 0
          for j in range(1,6):
              end = starter + increments
              train_subsets[i].append(filtered_train[i].select(range(starter, end)))
              starter = end

  for i in range(1,6):
    if refining_type in ["Increment_Training"]:
      sorted = []
      for j in range(num_classes):
          sorted.append(train_subsets[j][i-1])
      
      train_dataset = concatenate_datasets(sorted)
      train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
    # Model + Fine-Tuned Prep
    base = Augmented(copy.deepcopy(refer), num_classes=num_classes, embedding_space_dimension=embedding_space_dimension).to(device)

    optimizer = torch.optim.AdamW([
        {'params': base.model.parameters(), 'lr': 1e-5},
        {'params': base.classifier.parameters(), 'lr': 1e-3}
    ], weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * EPOCHS)

    if refiner == "Linear_Probe":
      for param in base.model.parameters():
        param.requires_grad = False
    elif refiner == "Reverse_Probe":
      for param in base.classifier.parameters():
        param.requires_grad = False

    best_val_loss = float('inf')
    best_epoch = -1

    for epoch in range(EPOCHS):
      print(f"Epoch {epoch}/{EPOCHS} - Best Val Loss: {best_val_loss:.4f}, (Epoch {best_epoch})")

      base.train()
      total_train_loss = 0
      train_steps = 0

      for batch in tqdm(train_loader, desc="Training"):
        optimizer.zero_grad()

        images = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits, _ = base(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        train_steps += 1

      avg_train_loss = total_train_loss / train_steps

      # Validation
      base.eval()
      correct = 0
      total = 0
      total_val_loss = 0
      val_steps = 0

      with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
          images = batch["pixel_values"].to(device, non_blocking=True)
          labels = batch["labels"].to(device, non_blocking=True)

          logits, _ = base(images)
          loss = criterion(logits, labels)

          preds = torch.argmax(logits, dim=1)
          correct += (preds == labels).sum().item()
          total += labels.size(0)

          total_val_loss += loss.item()
          val_steps += 1

      avg_val_loss = total_val_loss / val_steps
      val_acc = correct / total

      print(f"[Epoch {epoch}] Train Loss: {avg_train_loss} | Validation Loss: {avg_val_loss} | Validation Accuracy: {val_acc}")

      if epoch - best_epoch > 25:
        break
      if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch
        torch.save(base.state_dict(), f"{model_path}/best_{model_name}_{dataset_name[name]}_{refiner}_{i}.pt")

      scheduler.step()

In [ ]:
for name in range(0, len(dataset_name)):
    num_classes = num_class[name]
    results_path = f"./Results/{refining_type}/Refined_Accuracy/{refiner}"
    os.makedirs(results_path, exist_ok=True)

    def collate_fn(batch):
      images = torch.stack([example["pixel_values"] for example in batch])
      labels = torch.tensor([example["label"] for example in batch])
      
      return {
          "pixel_values": images,
          "labels": labels
      }
    
    model_path = f"./Models/{model_name}/{refining_type}/{refiner}/{dataset_name[name]}/best_{model_name}_{dataset_name[name]}_{refiner}"

    test = load_from_disk(f'./preprocessed/{dataset_name[name]}/test_processed')
    test.set_format(type='torch', columns=["image", "label", "pixel_values"])
    test_loader = DataLoader(test, batch_size=64, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
    
    base = Augmented(copy.deepcopy(refer), num_classes=num_classes, embedding_space_dimension=embedding_space_dimension).to(device).eval()
    fine_tuned = {}
    for i in range(1, 6):
        f_t = Augmented(copy.deepcopy(refer), num_classes=num_classes, embedding_space_dimension=embedding_space_dimension)
        f_t.load_state_dict(torch.load(f"{model_path}_{i}.pt", map_location=device))
        fine_tuned[i] = f_t.to(device).eval()
    
    correct_base = 0
    correct_fine_tuned = {i: 0 for i in range(1,6)}
    total = 0

    base_loss = 0
    fine_tuned_loss = {i: 0 for i in range(1,6)}
    total_loss = 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            images = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            total += labels.size(0)

            logits_base, _ = base(images)
            base_loss += criterion(logits_base, labels).item()
            pred = logits_base.argmax(dim=1)
            correct_base += (pred == labels).sum().item()

            for i in range(1,6):
                logits_fine_tuned, _ = fine_tuned[i](images)
                fine_tuned_loss[i] += criterion(logits_fine_tuned, labels).item()
                total_loss += 1
                pred = logits_fine_tuned.argmax(dim=1)
                correct_fine_tuned[i] += (pred == labels).sum().item()
    
    avg_base_loss = base_loss / total_loss
    accuracy_base = correct_base / total
    print(f"{dataset_name[name]} {refining_type} Results: {refiner}")
    print(f"\nAverage base loss: {avg_base_loss}, Base Accuracy: {accuracy_base}")
    acc_list = []
    for i in range(1,6):
        avg_best_loss = fine_tuned_loss[i] / total_loss
        accuracy_best = correct_fine_tuned[i] / total
        acc_list.append(accuracy_best)
        print(f"Average best loss: {avg_best_loss}, Best Accuracy: {accuracy_best}")
    print(f"{dataset_name[name]} = {acc_list}")

    data = {
        "Accuracy": [i for i in acc_list],
    }

    df = pd.DataFrame(data, index=[i for i in range(1,6)])
    df.to_json(f"{results_path}/{dataset_name[name]}_Accuracy.json", orient="records", indent=2)

In [ ]:
#  find /workspace -mindepth 1 -exec rm -rf {} +